# State Space Models (S4 & Mamba)

Companion notebook for the [State Space Models lesson](https://ml-viz-ruby.vercel.app/courses/rnns/04-state-space-models).

We implement a linear SSM both ways — as a **step-by-step recurrence** and as a **convolution** with
the SSM kernel — and verify they give *identical* outputs. Then we show the **O(L) vs O(L²)** scaling
advantage over attention. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## Intuition — an RNN you can train in parallel

RNNs are `O(1)` per token at inference but painfully **sequential** to train; attention trains in
parallel but costs `O(L²)`. **State Space Models** (S4, Mamba) get both: a *linear* recurrence
`h_t = A·h_{t−1} + B·u_t`, `y_t = C·h_t`. Because it's linear (no tanh between steps), the whole
sequence output equals a **convolution** with a precomputed kernel `K_j = C·Aʲ·B` — so training runs as
one parallel convolution while inference runs as a constant-memory recurrence. Same model, two modes.
S4 designs `A` for long memory; **Mamba** makes `B, C` input-dependent (selective), trading the
convolution for a parallel scan. We verify the duality and the scaling advantage.

## 1 — The linear recurrence (inference mode)

h_t = A h_{t-1} + B u_t,  y_t = C h_t. We roll it forward one step at a time — O(1) state update per
token, like an RNN but with a *linear* transition.

In [ ]:
# scalar state-space system for clarity (state dim 1)
A, B, C = 0.8, 0.5, 1.3
u = rng.normal(size=12)

def ssm_recurrent(A, B, C, u):
    h, ys = 0.0, []
    for ut in u:
        h = A * h + B * ut        # O(1) per step
        ys.append(C * h)
    return np.array(ys)

y_rec = ssm_recurrent(A, B, C, u)
print('recurrent output:', y_rec[:5].round(3), '...')

**What to notice:** the recurrence is an RNN with the tanh deleted — `h = A·h + B·u`, one `O(1)`
update per token, constant memory regardless of context length. That inference profile (no KV-cache
growing with length) is a huge deployment advantage over attention.

## 2 — The same thing as a convolution (training mode)

Because the recurrence is linear, y = K * u where the kernel is K_j = C·A^j·B. Computing the kernel
once and convolving processes the whole sequence in parallel — no step loop.

In [ ]:
def ssm_kernel(A, B, C, L):
    return np.array([C * (A ** j) * B for j in range(L)])

def ssm_conv(A, B, C, u):
    L = len(u)
    K = ssm_kernel(A, B, C, L)
    # causal convolution: y_t = sum_{j<=t} K_j u_{t-j}
    return np.array([np.dot(K[:t+1], u[t::-1]) for t in range(L)])

y_conv = ssm_conv(A, B, C, u)
print('kernel K:', ssm_kernel(A, B, C, 5).round(3), '...')
assert np.allclose(y_rec, y_conv)
print('\u2713 recurrence and convolution give IDENTICAL outputs (the SSM duality)')

**What to notice:** the assertion passes — the step-by-step recurrence and the one-shot convolution
produce **identical** outputs. That's the SSM duality: linearity lets you unroll
`y_t = Σ C·Aʲ·B·u_{t−j}` into a kernel, so training needs no sequential loop at all. The kernel values
`C·Aʲ·B` decay geometrically with `A = 0.8` — the model's memory profile is literally visible in `K`.

## The library way — the convolution via `scipy` (how it's really computed)

Our `ssm_conv` is an `O(L²)` loop; real SSM implementations compute the same causal convolution with
FFTs in `O(L log L)`. The cell verifies `scipy.signal.fftconvolve` reproduces both our convolution and
the recurrence.

In [ ]:
from scipy.signal import fftconvolve

K = ssm_kernel(A, B, C, len(u))
y_fft = fftconvolve(u, K)[:len(u)]          # causal convolution via FFT, O(L log L)
print('recurrent  :', y_rec[:4].round(4))
print('fftconvolve:', y_fft[:4].round(4))
assert np.allclose(y_fft, y_rec), "FFT convolution must equal the recurrence"
print('\nrecurrence == naive convolution == scipy fftconvolve ✓  (S4 trains via exactly this FFT path)')

**What to notice:** three routes, one answer — and the FFT route is the one S4 actually uses,
turning training into `O(L log L)` parallel work. The duality isn't a curiosity; it's the engineering
reason SSMs train as fast as CNNs while serving as cheaply as RNNs.

## 3 — O(L) SSM vs O(L²) attention

An SSM's cost grows linearly with sequence length; attention's grows quadratically. We count the
operations to make the scaling gap concrete.

In [ ]:
for L in [256, 1024, 4096, 16384]:
    attn = L * L          # all-pairs attention scores
    ssm = L               # one O(1) state update per token (recurrent inference)
    print(f'L={L:6d}:  attention ops ~ {attn:>12,}   SSM ops ~ {ssm:>8,}   ratio {attn/ssm:>8,.0f}x')
print('\nThe gap widens with length -> SSMs shine on very long sequences.')

**What to notice:** by 16k tokens, attention does **16,384× more work** per layer than an SSM's
recurrent mode — and the gap keeps widening quadratically. This is why SSMs excel on very long
sequences (audio, DNA, long documents) where attention's `L²` becomes prohibitive.

## Gotchas & tradeoffs

- **Stability requires `|A| < 1`** (eigenvalues inside the unit circle) — otherwise the kernel `C·Aʲ·B`
  and the state blow up. S4's structured `A` (HiPPO) is designed to sit close to 1 for long memory
  *without* exploding.
- **A plain linear SSM can't gate.** With fixed `A, B, C` the same dynamics apply to every token — no
  content-dependent behavior. **Mamba's selectivity** (input-dependent `B, C, Δ`) fixes this but breaks
  the convolution form, requiring a parallel **scan** instead.
- **Memory decays geometrically** in a scalar SSM — long memory needs either `A` near 1 (slow decay,
  near-unstable) or higher-dimensional structured state.
- **SSMs trade precision recall of specific tokens** (attention's strength) for efficient long-range
  aggregation — hybrids (attention + SSM layers) are common.

In [ ]:
# Stability: |A| < 1 decays, |A| > 1 explodes -- the kernel makes it visible
for A_test in [0.8, 0.99, 1.01]:
    K10 = ssm_kernel(A_test, B, C, 60)
    print(f'A={A_test}:  K[0]={K10[0]:.3f}  K[30]={K10[30]:.3f}  K[59]={K10[59]:.3f}'
          + ('   <- EXPLODES' if abs(K10[-1]) > abs(K10[0]) else ''))
print('\n-> memory length is set by how close |A| sits to 1; past 1 the system is unstable')

**What to notice:** at `A=0.8` the kernel dies within ~30 steps (short memory); at `A=0.99` it decays
slowly (long memory); at `A=1.01` it **grows without bound** — unstable. The whole design problem of S4
is parking `A`'s eigenvalues close to (but inside) the unit circle so memory is long *and* stable.

## Key takeaways

- An SSM is a **linear recurrence** `h = A·h + B·u`, `y = C·h` — an RNN without the nonlinearity
  between steps.
- Linearity ⇒ **duality**: the same model runs as an `O(1)`/token recurrence (inference) or a parallel
  convolution / FFT (`O(L log L)` training) — verified identical three ways.
- **`O(L)` vs attention's `O(L²)`**: SSMs win on very long sequences.
- Stability needs `|A| < 1`; long memory needs `A` near 1 (S4's structured `A`); content-dependence
  needs **Mamba's selective** `B, C` + parallel scan.

**Next:** the [course quiz](https://ml-viz-ruby.vercel.app/courses/rnns/05-quiz) — or jump ahead to
[Self-Attention](https://ml-viz-ruby.vercel.app/courses/transformers/01-self-attention).

## ✏️ Your turn

**Exercise.** Implement `ssm_step(A, B, C, h, u_t)` returning `(new_state, output)` for one
recurrence step, and `ssm_kernel_k(A, B, C, k)` returning the k-th convolution kernel coefficient
`C·A^k·B`. These are the two faces of the same linear SSM.

In [ ]:
def ssm_step(A, B, C, h, u_t):
    # TODO(you): return (new_state, output) = (A*h + B*u_t, C*new_state)
    return ...

def ssm_kernel_k(A, B, C, k):
    # TODO(you): the k-th kernel coefficient C * A^k * B
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
h, y0 = ssm_step(A, B, C, 0.0, u[0])
assert np.isclose(h, A*0.0 + B*u[0]) and np.isclose(y0, C*h)
assert np.isclose(ssm_kernel_k(A, B, C, 0), C*B)        # k=0 -> C*B
assert np.isclose(ssm_kernel_k(A, B, C, 2), C*A*A*B)
# rolling ssm_step matches the reference recurrence
hh, out = 0.0, []
for ut in u:
    hh, yt = ssm_step(A, B, C, hh, ut); out.append(yt)
assert np.allclose(out, y_rec)
print('\u2713 recurrence step and kernel coefficient are correct')

<details>
<summary>Solution</summary>

```python
def ssm_step(A, B, C, h, u_t):
    new_state = A * h + B * u_t
    return new_state, C * new_state

def ssm_kernel_k(A, B, C, k):
    return C * (A ** k) * B
```

The recurrence (constant memory, O(1)/token) is used for generation; the kernel (a parallel
convolution) is used for training. S4 chooses A cleverly for long memory; Mamba makes B, C depend on
the input for selectivity, trading the convolution for a parallel scan.

</details>